In [1]:
import numpy as np
import pandas as pd
import re
import ftfy
import html
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report
pd.set_option('display.max_colwidth', None)

In [2]:
df=pd.read_csv("development.csv",delimiter=",", index_col="Id")

### *Source* feature inspection

In [ ]:
n_nan = df['source'].isna().sum()
n_placeholder = (df['source'] == '\\N').sum()
n_empty = (df['source'].astype(str).str.strip() == '').sum()

print(f"NaN: {n_nan}")
print(f"Placeholder \\N: {n_placeholder}")
print(f"Stringhe vuote: {n_empty}")

df['source'] = df['source'].replace('\\N', np.nan)
df['source'] = df['source'].replace('', np.nan)
df['source'] = df['source'].fillna('Other')

min_freq = 5  
source_counts = df['source'].value_counts()
sources_kept = source_counts[source_counts >= min_freq].index.tolist()

df['source'] = np.where(df['source'].isin(sources_kept),df['source'],'Other')

final_source_counts = df['source'].value_counts()

print(f"Numero categorie finali (incluso Other): {len(final_source_counts)}")

NaN: 0
Placeholder \N: 294
Stringhe vuote: 0
Numero categorie finali (incluso Other): 489


### *Title* feature inspection

In [4]:
n_nan_title = df['title'].isna().sum()
n_placeholders_title = (df['title']== '\\N').sum()
n_empty_title = (df['title'].astype(str).str.strip()=='').sum()

print(f"Number of NaN rows: {n_nan_title}")
print(f"Number of placeholders (\\N): {n_placeholders_title}")
print(f"Number of empty rows: {n_empty_title}")
print("Titles Sample:")
print(df['title'].sample(10))

Number of NaN rows: 1
Number of placeholders (\N): 0
Number of empty rows: 2
Titles Sample:
Id
12465               Koizumi dismisses Chinese criticism
8862                    IT managers hate buggy software
78401      Myanmar troops fire shots to disperse crowds
25328                  Laws planned to nationalise Rock
77958      Former Novell head Messman quits board early
34416               Making Money Fast on Very Slow Cars
49251                Brown unveils welfare reform plans
43794    Mother: Scott Peterson a 'kind, gentle person'
39758                 Hutton call for Brown 'challenge'
29755                       All Eyes on OPEC's Actions 
Name: title, dtype: object


### *Article* feature inspection

In [5]:
n_nan_article = df['article'].isna().sum()
n_placeholders_article = (df['article']=='\\N').sum()
n_empty_article = (df['article'].astype(str).str.strip()=='').sum()

print(f"Number of NaN rows: {n_nan_article}")
print(f"Number of placeholders (\\N): {n_placeholders_article}")
print(f"Number of empty rows: {n_empty_article}")
print("Articles Sample")
print(df['article'].sample(10))

Number of NaN rows: 1
Number of placeholders (\N): 1874
Number of empty rows: 7
Articles Sample
Id
74075                                                                                                                                                                                          <p><a href="http://us.rd.yahoo.com/dailynews/rss/us/*http://news.yahoo.com/s/ap/20070525/ap_on_re_us/holiday_travel"><img src="http://d.yimg.com/us.yimg.com/p/ap/20070523/capt.8754adf6ae4343beb13cf4c8ccf08a8b.gas_prices_dchg102.jpg?x=130&y=72&sig=ZoVnergrP6bmTX7EgSg8Nw--" align="left" height="72" width="130" alt="Gas prices in the northwest section of the District of Columbia are displayed at this Exxon service station Wednesday, May 23, 2007, in Washington. (AP Photo/Haraz N. Ghanbari)" border="0" /></a>AP - This is getaway day for what's looked at as the unofficial start of summer. Millions of people will be climbing into their cars or getting on airplanes for the Memorial Day weekend.</p><br clear=

### *PageRank* feature inspection

In [6]:
n_nan_pr = df['page_rank'].isna().sum()
n_placeholders_pr = (df['page_rank']=='\\N').sum()
n_empty_pr = (df['page_rank'].astype(str).str.strip()=='').sum()
rank_5=np.array([df['page_rank']==5]).sum()

print(f"Number of NaN rows: {n_nan_pr}")
print(f"Number of placeholders (\\N): {n_placeholders_pr}")
print(f"Number of empty rows: {n_empty_pr}")
print(f"Number of articles with PageRank 5: {rank_5}")

Number of NaN rows: 0
Number of placeholders (\N): 0
Number of empty rows: 0
Number of articles with PageRank 5: 73891


### *Timestamp* feature inspection 

In [7]:
n_nan_time = df['timestamp'].isna().sum()
n_placeholders_time = (df['timestamp']=='\\N').sum()
n_empty_time = (df['timestamp'].astype(str).str.strip()=='').sum()
n_uslesess_time=np.array([df['timestamp']=="0000-00-00 00:00:00"]).sum()

print(f"Number of NaN rows: {n_nan_time}")
print(f"Number of placeholders (\\N): {n_placeholders_time}")
print(f"Number of empty rows: {n_empty_time}")
print(f"Number invalid dates (0000-00-00 00:00:00): {n_uslesess_time}")
print(df['timestamp'].sample(10))

Number of NaN rows: 0
Number of placeholders (\N): 0
Number of empty rows: 0
Number invalid dates (0000-00-00 00:00:00): 27750
Id
41969    0000-00-00 00:00:00
11022    2007-07-20 08:19:03
61385    0000-00-00 00:00:00
26369    2007-10-30 08:08:14
78716    0000-00-00 00:00:00
50271    2007-06-11 19:09:47
28938    2007-09-21 16:23:22
17875    2004-10-10 23:23:45
71570    2007-07-14 17:06:48
19619    2006-12-15 00:36:33
Name: timestamp, dtype: object


### *Timestamp* feature processing

In [8]:
def process_timestamp(df):
    df = df.copy()

    dt = pd.to_datetime(df['timestamp'], errors='coerce')

    df['has_date'] = dt.notna().astype(int)
    df['quarter'] = dt.dt.quarter.fillna(-1).astype(int)
    df['is_weekend'] = dt.dt.dayofweek.isin([5, 6]).fillna(False).astype(int)

    df = df.drop(columns=['timestamp'])
    return df

df = process_timestamp(df)

timestamp_cols = ['has_date', 'is_weekend','quarter']
print(f"Timestamp expanded columns: {timestamp_cols}")
print("'timestamp' raw column deleted")
print("Sample of 10 timestamps:")
print(df[timestamp_cols].sample(10))

Timestamp expanded columns: ['has_date', 'is_weekend', 'quarter']
'timestamp' raw column deleted
Sample of 10 timestamps:
       has_date  is_weekend  quarter
Id                                  
41094         1           0        3
48096         1           1        4
37718         1           0        4
39745         1           0        1
39100         1           1        1
22194         1           0        3
12576         0           0       -1
10110         1           0        1
20210         1           0        1
65541         0           0       -1


### *Title* feature stemming

In [9]:
def clean_text_light(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = ftfy.fix_text(text)
    text = html.unescape(text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.\,\-\%\$\€\£]", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def process_title_final(df):
    df = df.copy()

    df['title_clean'] = df['title'].apply(clean_text_light)
    df = df.drop(columns=['title'])

    return df


df = process_title_final(df)

print("Sample of 10 cleaned titles:")
print(df['title_clean'].sample(10).to_string(index=False))


Sample of 10 cleaned titles:
Id
                            fullstop-size cure for cataract
                            a horse bet that needs a payoff
                   in britain, larger lead for labour party
                progess not trophies the key for wenger afp
              shoppers loved gadgets no more than last year
                                           stocks step back
               early voters transform campaign landscape ap
indian, british scientists working on drug to fight malaria
               minaya ahead in count even if he strikes out
                        jeffersons star at target center ap


### *Article* feature stemming

In [10]:
##TODO remove comments and change variables, also for title above
def clean_text_light(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = ftfy.fix_text(text)
    text = html.unescape(text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)

    text = re.sub(r'\b(a\s+href|href|img\s+src|nbsp|read\s+more|click\s+here)\b',' ',text,flags=re.IGNORECASE)

    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.\,\-\%\$\€\£]", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def process_article_final(df):
    df = df.copy()

    df['article'] = df['article'].replace('\\N', '').fillna('').astype(str)
    df['article_clean'] = df['article'].apply(clean_text_light)
    df = df.drop(columns=['article'])

    return df

df = process_article_final(df)

print("Sample of 10 cleaned articles:")
print(df['article_clean'].sample(10).to_string(index=False))

Sample of 10 cleaned articles:
Id
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [11]:
df['text'] = (df['title_clean'] + ' ' + df['title_clean'] + ' [TITLE] ' + df['article_clean']).str.strip()
df = df.drop(columns=['title_clean', 'article_clean'])

### Encoding categorical features

In [13]:
TEXT_COL = "text"
CAT_COLS = ["source"]
NUM_COLS = ["page_rank", "has_date", "is_weekend", "quarter"]
TARGET_COL = "label"

X = df[[TEXT_COL] + CAT_COLS + NUM_COLS].copy()
y = df[TARGET_COL].copy()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.95,
            max_features=150_000,
            sublinear_tf=True  
        ), TEXT_COL),
        ("source_ohe", OneHotEncoder(
            handle_unknown="infrequent_if_exist",
            min_frequency=5,
            sparse_output=True
        ), CAT_COLS),
        ("num", "passthrough", NUM_COLS),
    ],
    remainder="drop"
)

svm = LinearSVC(
    class_weight="balanced",
    random_state=42
)

pipe = Pipeline([
    ("prep", preprocess),
    ("clf", svm)
])

param_dist = {
    "clf__C": np.logspace(-3, 1, 10),  
    "clf__loss": ["hinge", "squared_hinge"]
}

search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=10,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)

best_model = search.best_estimator_
print("Best params:", search.best_params_)
print("Best CV Macro F1:", search.best_score_)

y_val_pred = best_model.predict(X_val)
print("Validation Macro F1:", f1_score(y_val, y_val_pred, average="macro"))
print(classification_report(y_val, y_val_pred, digits=4))


/Users/giorgiozoccatelli/miniforge3/envs/data/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/giorgiozoccatelli/miniforge3/envs/data/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/giorgiozoccatelli/miniforge3/envs/data/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/giorgiozoccatelli/miniforge3/envs/data/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/giorgiozoccatelli/miniforge3/envs/data/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iteration

Best params: {'clf__loss': 'squared_hinge', 'clf__C': 0.1668100537200059}
Best CV Macro F1: 0.6934341319897698
Validation Macro F1: 0.7012830263696089
              precision    recall  f1-score   support

           0     0.7391    0.7473    0.7432      4709
           1     0.7286    0.7998    0.7625      2118
           2     0.8192    0.8118    0.8155      2232
           3     0.6097    0.5028    0.5511      1995
           4     0.7731    0.9376    0.8474      1715
           5     0.5717    0.4611    0.5105      2611
           6     0.5809    0.8161    0.6787       620

    accuracy                         0.7091     16000
   macro avg     0.6889    0.7252    0.7013     16000
weighted avg     0.7030    0.7091    0.7026     16000

